# Notebook 3: Transition Evidence — Endometrial Hyperplasia DEG Analysis (GSE106191)
## PMOS-Cancer BridgeGraph — Phase 3 (Blueprint v2)

**Goal:** Identify differentially expressed genes between endometrial hyperplasia and endometrial cancer using limma with empirical-Bayes moderation. Output feeds into **T(g)** (Transition) component of PCBS.

**Dataset:** GSE106191 — Hermens et al.
- 64 endometrial cancer + 33 endometrial hyperplasia samples
- **No normal tissue samples** — this dataset captures the hyperplasia-to-cancer transition
- Platform: Affymetrix HG-U133 Plus 2.0 (GPL570)
- Comparison: **Hyperplasia vs Cancer** (transition/progression genes)

**Why hyperplasia-vs-cancer?**  
GSE106191 has zero normal endometrium samples. The hyperplasia-vs-cancer comparison captures genes that change during the pre-cancer → cancer transition — exactly what the T(g) Transition component of PCBS needs. Published papers (Frontiers in Genetics 2021, IJWH 2024) validate this approach.

**Key changes from old pipeline:**
- limma (empirical-Bayes) instead of Welch's t-test
- GEOparse for proper probe-to-gene mapping
- Adjusted p-value < 0.05 only — no hard logFC cutoff
- All genes retained for downstream PCBS scoring

## 1. Setup and Installation

In [ ]:
# Install required packages
!pip install -q GEOparse gseapy rpy2 adjustText

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import os

os.makedirs('results', exist_ok=True)
os.makedirs('figures', exist_ok=True)

print('Setup complete.')

## 2. Download and Parse GSE106191 using GEOparse

GEOparse handles downloading the dataset AND the platform annotation (GPL570) properly — this fixes the probe mapping bug from the old pipeline where NCBI FTP returned 404.

In [ ]:
import GEOparse

# Download GSE106191 (may take a few minutes — large dataset)
print('Downloading GSE106191 from GEO...')
print('(This includes expression data + GPL570 platform annotation)')
gse = GEOparse.get_GEO(geo='GSE106191', destdir='/tmp')

print(f'\nDataset: {gse.metadata["title"][0]}')
print(f'Platform: {list(gse.gpls.keys())}')
print(f'Total samples: {len(gse.gsms)}')

## 3. Extract Sample Metadata and Classify Groups

GSE106191 contains **hyperplasia** and **cancer** samples. We need to parse the sample characteristics to classify each sample correctly.

In [ ]:
# Extract metadata from each GSM
metadata = []
for gsm_name, gsm in gse.gsms.items():
    title = gsm.metadata['title'][0]
    
    # Parse characteristics
    chars = {}
    for ch in gsm.metadata.get('characteristics_ch1', []):
        if ':' in ch:
            key, val = ch.split(':', 1)
            chars[key.strip().lower()] = val.strip()
    
    metadata.append({
        'GSM': gsm_name,
        'Title': title,
        'characteristics': chars
    })

meta_df = pd.DataFrame(metadata)

# Show all characteristic keys to understand the structure
all_keys = set()
for chars in meta_df['characteristics']:
    all_keys.update(chars.keys())
print('Available characteristics:', sorted(all_keys))

# Show a few examples
print('\nSample metadata examples:')
for i in range(min(5, len(meta_df))):
    row = meta_df.iloc[i]
    print(f"  {row['GSM']}: {row['Title']}")
    for k, v in row['characteristics'].items():
        print(f"    {k}: {v}")

In [ ]:
# Classify samples into Hyperplasia vs Cancer
# The exact field names depend on the dataset — we check the most common ones

def classify_sample(chars, title):
    """
    Classify a sample as Hyperplasia, Cancer, or Unknown.
    Checks characteristics fields and title for keywords.
    """
    # Combine all text sources
    text = title.lower()
    for v in chars.values():
        text += ' ' + str(v).lower()
    
    # Check for hyperplasia indicators
    hyperplasia_kw = ['hyperplasia', 'hyperplastic', 'eah', 'eh ',
                      'atypical hyperplasia', 'complex hyperplasia',
                      'simple hyperplasia', 'endometrial intraepithelial neoplasia',
                      'ein', 'premalignant', 'precancer']
    for kw in hyperplasia_kw:
        if kw in text:
            return 'Hyperplasia'
    
    # Check for cancer/carcinoma indicators
    cancer_kw = ['carcinoma', 'cancer', 'tumor', 'tumour', 'malignant',
                 'endometrioid', 'serous', 'clear cell', 'adenocarcinoma',
                 'ec ', 'grade', 'stage', 'figo']
    for kw in cancer_kw:
        if kw in text:
            return 'Cancer'
    
    return 'Unknown'


meta_df['Group'] = meta_df.apply(
    lambda row: classify_sample(row['characteristics'], row['Title']), axis=1
)

print('Sample classification:')
print(meta_df['Group'].value_counts())

# Show unknown samples if any, so we can debug
unknown = meta_df[meta_df['Group'] == 'Unknown']
if len(unknown) > 0:
    print(f'\n⚠ {len(unknown)} Unknown samples — inspect manually:')
    for _, row in unknown.iterrows():
        print(f"  {row['GSM']}: {row['Title']}")
        for k, v in row['characteristics'].items():
            print(f"    {k}: {v}")
else:
    print('\n✓ All samples classified.')

In [ ]:
# If the automatic classifier didn't work perfectly, inspect and fix.
# This cell lets you manually review and override if needed.

# Show distribution
print('Final group assignment:')
for grp in ['Cancer', 'Hyperplasia', 'Unknown']:
    n = (meta_df['Group'] == grp).sum()
    if n > 0:
        print(f'  {grp}: {n} samples')

# Filter to only Cancer and Hyperplasia
meta_df = meta_df[meta_df['Group'].isin(['Cancer', 'Hyperplasia'])].copy()
meta_df = meta_df.reset_index(drop=True)

cancer_samples = meta_df[meta_df['Group'] == 'Cancer']['GSM'].tolist()
hyp_samples = meta_df[meta_df['Group'] == 'Hyperplasia']['GSM'].tolist()

print(f'\nUsing: {len(cancer_samples)} Cancer + {len(hyp_samples)} Hyperplasia = {len(meta_df)} total')
print(f'\nExpected: ~64 Cancer + ~33 Hyperplasia = ~97 total')

## 4. Extract Expression Matrix and Map Probes to Gene Symbols

GPL570 (Affymetrix HG-U133 Plus 2.0) has ~54,675 probes. We use the GEOparse-provided platform annotation to map probes → gene symbols, then collapse duplicate probes per gene by keeping the probe with highest mean expression.

In [ ]:
# Build expression matrix
expr_dict = {}
for gsm_name in meta_df['GSM']:
    gsm = gse.gsms[gsm_name]
    table = gsm.table
    expr_dict[gsm_name] = table.set_index('ID_REF')['VALUE'].astype(float)

expr = pd.DataFrame(expr_dict)
expr.index = expr.index.astype(str)

print(f'Raw expression matrix: {expr.shape[0]} probes x {expr.shape[1]} samples')
print(f'Value range: {expr.min().min():.2f} to {expr.max().max():.2f}')
print(f'Missing values: {expr.isnull().sum().sum()}')

# Drop rows with all NaN
expr = expr.dropna(how='all')
print(f'After removing empty rows: {expr.shape[0]} probes')

In [ ]:
# Get platform annotation for probe-to-gene mapping
gpl_name = list(gse.gpls.keys())[0]
gpl = gse.gpls[gpl_name]

print(f'Platform: {gpl_name}')
print(f'Annotation table: {gpl.table.shape[0]} probes')
print(f'Annotation columns: {list(gpl.table.columns)}')

# Find gene symbol column — GPL570 typically uses 'Gene Symbol'
gene_col = None
for col in gpl.table.columns:
    col_lower = col.lower().replace(' ', '_')
    if col_lower in ['gene_symbol', 'gene symbol', 'symbol', 'genesymbol']:
        gene_col = col
        break

if gene_col is None:
    # Broader search
    for col in gpl.table.columns:
        if 'gene' in col.lower() and 'symbol' in col.lower():
            gene_col = col
            break

if gene_col is None:
    # Try gene_assignment (parse symbol from it)
    for col in gpl.table.columns:
        if 'gene_assignment' in col.lower():
            gene_col = col
            break

print(f'\nUsing column for gene mapping: {gene_col}')
print(f'\nSample mappings:')
print(gpl.table[['ID', gene_col]].dropna().head(10).to_string())

In [ ]:
# Create probe-to-gene mapping
annot = gpl.table[['ID', gene_col]].copy()
annot['ID'] = annot['ID'].astype(str)

# Handle gene_assignment parsing if needed
if 'assignment' in gene_col.lower():
    def extract_symbol(val):
        if pd.isna(val) or val == '---' or val == '':
            return None
        parts = str(val).split('//')
        if len(parts) >= 2:
            return parts[1].strip()
        return None
    annot['Gene'] = annot[gene_col].apply(extract_symbol)
else:
    annot['Gene'] = annot[gene_col].astype(str)

# Clean up — remove empty, '---', NaN, and multi-mapped probes (take first symbol)
annot = annot.dropna(subset=['Gene'])
annot = annot[~annot['Gene'].isin(['---', '', 'nan', 'None'])]

# Some GPL570 entries have multiple symbols separated by '///'
annot['Gene'] = annot['Gene'].str.split('///').str[0].str.strip()
annot = annot[annot['Gene'] != '']

probe_to_gene = dict(zip(annot['ID'], annot['Gene']))
print(f'Probe-to-gene mapping: {len(probe_to_gene)} probes have gene symbols')

# Map probes to genes
expr['Gene'] = expr.index.map(probe_to_gene)

n_before = len(expr)
expr = expr.dropna(subset=['Gene'])
print(f'Mapped: {len(expr)}/{n_before} probes have gene symbols')

# Collapse duplicate probes per gene — keep the one with highest mean expression
sample_cols = [c for c in expr.columns if c != 'Gene']
expr['mean_expr'] = expr[sample_cols].mean(axis=1)
expr = expr.sort_values('mean_expr', ascending=False)
expr = expr.drop_duplicates(subset='Gene', keep='first')
expr = expr.drop('mean_expr', axis=1)
expr = expr.set_index('Gene')

print(f'\nFinal: {expr.shape[0]} unique genes x {expr.shape[1]} samples')

# Verify known genes are present
check_genes = ['ESR1', 'PTEN', 'PIK3CA', 'AKT1', 'TP53', 'MMP9', 'CTNNB1',
               'KRAS', 'PGR', 'AR', 'SERPINE1', 'VEGFA', 'EGFR', 'MYC']
found = [g for g in check_genes if g in expr.index]
print(f'\nKnown gene check: {len(found)}/{len(check_genes)} found')
print(f'  Found: {found}')
print(f'  Missing: {[g for g in check_genes if g not in expr.index]}')

## 5. Quality Control

In [ ]:
# Check if values are already log-transformed
print('Expression value summary:')
print(f'  Min: {expr.min().min():.2f}')
print(f'  Max: {expr.max().max():.2f}')
print(f'  Median: {expr.median().median():.2f}')

# If max > 25, likely raw intensities — need log2 transform
if expr.max().max() > 25:
    print('\n⚠ Values appear to be raw intensities — applying log2 transform')
    expr = np.log2(expr.clip(lower=1))  # clip to avoid log(0)
    print(f'  After log2: range {expr.min().min():.2f} to {expr.max().max():.2f}')
else:
    print('\n✓ Values appear to be log-transformed already.')

In [ ]:
# Distribution of expression values per sample, colored by group
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Reorder by group for visual clarity
ordered_gsms = meta_df.sort_values('Group')['GSM'].tolist()
ordered_groups = meta_df.sort_values('Group')['Group'].tolist()

colors = ['#e74c3c' if g == 'Cancer' else '#9b59b6' for g in ordered_groups]

bp = axes[0].boxplot([expr[col].dropna().values for col in ordered_gsms],
                      patch_artist=True, showfliers=False)
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color)
axes[0].set_xticklabels(['' for _ in ordered_gsms])  # Too many to label
axes[0].set_ylabel('Expression (log2)')
axes[0].set_title('Expression per sample (red=Cancer, purple=Hyperplasia)')

# Density by group
for grp, color, label in [('Cancer', '#e74c3c', 'Cancer'),
                           ('Hyperplasia', '#9b59b6', 'Hyperplasia')]:
    grp_gsms = meta_df[meta_df['Group'] == grp]['GSM'].tolist()
    vals = expr[grp_gsms].values.flatten()
    vals = vals[~np.isnan(vals)]
    axes[1].hist(vals, bins=80, alpha=0.4, color=color, density=True, label=label)
axes[1].set_xlabel('Expression (log2)')
axes[1].set_ylabel('Density')
axes[1].set_title('Expression density by group')
axes[1].legend()

plt.tight_layout()
plt.savefig('figures/01_qc_distributions_gse106191.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# PCA
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

X = expr[meta_df['GSM'].tolist()].T.values
X_scaled = StandardScaler().fit_transform(X)

pca = PCA(n_components=5)
pcs = pca.fit_transform(X_scaled)

pca_df = pd.DataFrame({
    'PC1': pcs[:, 0],
    'PC2': pcs[:, 1],
    'Group': meta_df['Group'].values
})

fig, ax = plt.subplots(figsize=(8, 6))
for grp, color, marker in [('Cancer', '#e74c3c', 'o'),
                            ('Hyperplasia', '#9b59b6', 's')]:
    mask = pca_df['Group'] == grp
    ax.scatter(pca_df.loc[mask, 'PC1'], pca_df.loc[mask, 'PC2'],
               c=color, marker=marker, label=grp, s=60,
               edgecolors='black', linewidth=0.5, alpha=0.7)

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_title('PCA — GSE106191 (Hyperplasia vs Cancer)')
ax.legend()

plt.tight_layout()
plt.savefig('figures/02_pca_gse106191.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Variance explained: PC1={pca.explained_variance_ratio_[0]*100:.1f}%, '
      f'PC2={pca.explained_variance_ratio_[1]*100:.1f}%, '
      f'PC3={pca.explained_variance_ratio_[2]*100:.1f}%')

## 6. Differential Expression Analysis using limma

We compare **Hyperplasia vs Cancer** to find genes that change during the transition from pre-cancer to cancer. This is the T(g) evidence for PCBS.

- **Method:** limma with empirical-Bayes moderation (gold standard for microarray)
- **Direction convention:** logFC > 0 → higher in Hyperplasia than Cancer (i.e. downregulated during cancer progression); logFC < 0 → higher in Cancer (i.e. upregulated during progression)
- **Filter:** adj.p < 0.05 only — no hard logFC cutoff

In [ ]:
# Setup R environment for limma
import rpy2.robjects as ro
from rpy2.robjects import pandas2ri, r
from rpy2.robjects.packages import importr

pandas2ri.activate()

# Install limma if needed
r('''
if (!requireNamespace("limma", quietly = TRUE)) {
    if (!requireNamespace("BiocManager", quietly = TRUE))
        install.packages("BiocManager")
    BiocManager::install("limma")
}
library(limma)
''')

print('limma loaded successfully in R.')

In [ ]:
def run_limma(expr_df, group1_ids, group2_ids,
              group1_name='Hyperplasia', group2_name='Cancer'):
    """
    Run limma DEG analysis via rpy2.
    
    Convention: positive logFC = higher in group1 (Hyperplasia),
    negative logFC = higher in group2 (Cancer).
    
    Uses standard eBayes() — no trend=TRUE since this is microarray data.
    
    Returns DataFrame with Gene, logFC, adj_pvalue, Direction for every gene.
    """
    # Prepare expression matrix (genes x samples in correct order)
    all_samples = group1_ids + group2_ids
    expr_sub = expr_df[all_samples].copy()
    
    # Create design labels
    groups = [group1_name] * len(group1_ids) + [group2_name] * len(group2_ids)
    
    # Transfer to R
    r_expr = pandas2ri.py2rpy(expr_sub)
    ro.globalenv['expr_matrix'] = r_expr
    ro.globalenv['groups'] = ro.StrVector(groups)
    ro.globalenv['g1_name'] = group1_name
    ro.globalenv['g2_name'] = group2_name
    
    # Run limma
    r('''
    groups_factor <- factor(groups, levels = c(g2_name, g1_name))
    design <- model.matrix(~ groups_factor)
    colnames(design) <- c("Intercept", paste0(g1_name, "_vs_", g2_name))
    
    fit <- lmFit(as.matrix(expr_matrix), design)
    fit <- eBayes(fit)
    
    results <- topTable(fit, coef = 2, number = Inf, sort.by = "none",
                        adjust.method = "BH")
    ''')
    
    # Get results back to Python
    r_results = ro.globalenv['results']
    results_df = pandas2ri.rpy2py(r_results)
    results_df['Gene'] = expr_sub.index.tolist()
    
    results_df = results_df.rename(columns={
        'P.Value': 'pvalue',
        'adj.P.Val': 'adj_pvalue'
    })
    
    # Direction: adj.p < 0.05 only — no logFC cutoff
    results_df['Direction'] = 'NS'
    sig_mask = results_df['adj_pvalue'] < 0.05
    results_df.loc[sig_mask & (results_df['logFC'] > 0), 'Direction'] = 'Up_in_Hyperplasia'
    results_df.loc[sig_mask & (results_df['logFC'] < 0), 'Direction'] = 'Up_in_Cancer'
    
    results_df = results_df.sort_values('pvalue')
    
    n_hyp = (results_df['Direction'] == 'Up_in_Hyperplasia').sum()
    n_can = (results_df['Direction'] == 'Up_in_Cancer').sum()
    n_total = n_hyp + n_can
    print(f'  DEGs (adj.p < 0.05): {n_hyp} up in hyperplasia + {n_can} up in cancer = {n_total} total')
    print(f'  Total genes tested: {len(results_df)}')
    
    return results_df

print('limma DEG function ready.')

In [ ]:
# Run main DEG analysis: Hyperplasia vs Cancer
print(f'=== Hyperplasia vs Cancer: {len(hyp_samples)} Hyperplasia vs {len(cancer_samples)} Cancer ===')
deg_transition = run_limma(expr, hyp_samples, cancer_samples)

print(f'\nlogFC distribution of significant DEGs:')
sig = deg_transition[deg_transition['Direction'] != 'NS']
if len(sig) > 0:
    print(f'  Mean |logFC|: {sig["logFC"].abs().mean():.3f}')
    print(f'  Median |logFC|: {sig["logFC"].abs().median():.3f}')
    print(f'  Max |logFC|: {sig["logFC"].abs().max():.3f}')
    print(f'  Genes with |logFC| > 1: {(sig["logFC"].abs() > 1).sum()}')
    print(f'  Genes with |logFC| > 0.5: {(sig["logFC"].abs() > 0.5).sum()}')
    print(f'  Genes with |logFC| < 0.3: {(sig["logFC"].abs() < 0.3).sum()}')
else:
    print('  No significant DEGs found.')

## 7. Volcano Plot

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

color_map = {'Up_in_Hyperplasia': '#9b59b6', 'Up_in_Cancer': '#e74c3c', 'NS': '#cccccc'}
label_map = {'Up_in_Hyperplasia': 'Higher in Hyperplasia',
             'Up_in_Cancer': 'Higher in Cancer',
             'NS': 'Not Significant'}

for direction in ['NS', 'Up_in_Hyperplasia', 'Up_in_Cancer']:
    mask = deg_transition['Direction'] == direction
    ax.scatter(deg_transition.loc[mask, 'logFC'],
               -np.log10(deg_transition.loc[mask, 'adj_pvalue'].clip(lower=1e-50)),
               c=color_map[direction], s=8, alpha=0.5, label=label_map[direction])

# Label top 15 genes
top_genes = deg_transition[deg_transition['Direction'] != 'NS'].sort_values('adj_pvalue').head(15)
for _, row in top_genes.iterrows():
    ax.annotate(row['Gene'],
                (row['logFC'], -np.log10(max(row['adj_pvalue'], 1e-50))),
                fontsize=7, alpha=0.85, ha='center',
                bbox=dict(boxstyle='round,pad=0.15', fc='white', ec='gray', alpha=0.6))

ax.axhline(-np.log10(0.05), color='grey', linestyle='--', linewidth=0.5, alpha=0.5)
ax.set_xlabel('log2 Fold Change (Hyperplasia vs Cancer)', fontsize=12)
ax.set_ylabel('-log10(adj. p-value)', fontsize=12)
ax.set_title('Volcano Plot — Hyperplasia vs Cancer Transition (GSE106191)', fontsize=13)
ax.legend(fontsize=9, loc='upper right')

plt.tight_layout()
plt.savefig('figures/03_volcano_gse106191.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Pathway Enrichment (GO + KEGG)

We run enrichment separately for genes upregulated in hyperplasia (lost during cancer progression) and genes upregulated in cancer (gained during progression).

In [ ]:
import gseapy as gp

def run_enrichment(gene_list, name, max_genes=500):
    """Run GO and KEGG enrichment on a gene list."""
    genes = [str(g) for g in gene_list[:max_genes]]
    
    if len(genes) < 5:
        print(f'  {name}: Only {len(genes)} genes, skipping.')
        return None
    
    print(f'  {name}: Enriching {len(genes)} genes...')
    
    try:
        enr = gp.enrichr(
            gene_list=genes,
            gene_sets=['GO_Biological_Process_2023', 'KEGG_2021_Human'],
            organism='human',
            outdir=None,
            no_plot=True
        )
        results = enr.results
        sig = results[results['Adjusted P-value'] < 0.05].sort_values('Adjusted P-value')
        sig.to_csv(f'results/enrichment_{name}.csv', index=False)
        
        n_go = len(sig[sig['Gene_set'] == 'GO_Biological_Process_2023'])
        n_kegg = len(sig[sig['Gene_set'] == 'KEGG_2021_Human'])
        print(f'    Significant: {n_go} GO terms, {n_kegg} KEGG pathways')
        return sig
    except Exception as e:
        print(f'    Enrichment error: {e}')
        return None


# Genes higher in hyperplasia (lost during cancer progression)
print('=== Enrichment: Higher in Hyperplasia (lost in cancer) ===')
hyp_genes = (deg_transition[deg_transition['Direction'] == 'Up_in_Hyperplasia']
             .sort_values('adj_pvalue')['Gene'].tolist())
enr_hyp = run_enrichment(hyp_genes, 'transition_higher_in_hyperplasia')

# Genes higher in cancer (gained during progression)
print('\n=== Enrichment: Higher in Cancer (gained in progression) ===')
can_genes = (deg_transition[deg_transition['Direction'] == 'Up_in_Cancer']
             .sort_values('adj_pvalue')['Gene'].tolist())
enr_can = run_enrichment(can_genes, 'transition_higher_in_cancer')

In [ ]:
# Plot top enriched terms
def plot_top_terms(enr_df, title, gene_set_filter=None, top_n=15):
    if enr_df is None or len(enr_df) == 0:
        print(f'No significant terms for: {title}')
        return
    
    df = enr_df.copy()
    if gene_set_filter:
        df = df[df['Gene_set'] == gene_set_filter]
    
    df = df.head(top_n)
    if len(df) == 0:
        print(f'No terms for: {title}')
        return
    
    df['Term_short'] = df['Term'].str[:60]
    
    fig, ax = plt.subplots(figsize=(10, max(3, len(df) * 0.35)))
    ax.barh(range(len(df)), -np.log10(df['Adjusted P-value']),
            color='#2ecc71', edgecolor='black', linewidth=0.3)
    ax.set_yticks(range(len(df)))
    ax.set_yticklabels(df['Term_short'], fontsize=8)
    ax.set_xlabel('-log10(Adjusted P-value)')
    ax.set_title(title, fontsize=10)
    ax.invert_yaxis()
    plt.tight_layout()
    safe_title = title.replace(' ', '_').replace('/', '_').replace('(', '').replace(')', '').replace('—', '-')
    plt.savefig(f'figures/04_enrich_{safe_title[:60]}.png', dpi=150, bbox_inches='tight')
    plt.show()


plot_top_terms(enr_hyp, 'Higher in Hyperplasia — GO BP', 'GO_Biological_Process_2023')
plot_top_terms(enr_hyp, 'Higher in Hyperplasia — KEGG', 'KEGG_2021_Human')
plot_top_terms(enr_can, 'Higher in Cancer — GO BP', 'GO_Biological_Process_2023')
plot_top_terms(enr_can, 'Higher in Cancer — KEGG', 'KEGG_2021_Human')

## 9. Validate Known PMOS–Cancer Transition Genes

Check whether genes known from literature to be involved in the hyperplasia→cancer transition are captured by this analysis.

In [ ]:
# Known genes involved in endometrial hyperplasia → cancer transition
known_transition = {
    'PTEN': 'Tumor suppressor — lost early in hyperplasia→cancer',
    'PIK3CA': 'PI3K pathway — mutated in EC',
    'KRAS': 'RAS oncogene — mutated in Type I EC',
    'CTNNB1': 'Beta-catenin — Wnt pathway mutations in EC',
    'TP53': 'Tumor suppressor — mutated in aggressive EC',
    'ARID1A': 'Chromatin remodeling — mutated in EC',
    'ESR1': 'Estrogen receptor — drives proliferation',
    'PGR': 'Progesterone receptor — protective, often lost in cancer',
    'MMP2': 'Matrix metalloproteinase — invasion',
    'MMP9': 'Matrix metalloproteinase — invasion',
    'VEGFA': 'Angiogenesis — tumor vasculature',
    'MYC': 'Oncogene — proliferation driver',
    'EGFR': 'EGF receptor — growth signaling',
    'AKT1': 'PI3K/AKT — survival signaling',
    'ERBB2': 'HER2 — growth factor receptor',
    'CDH1': 'E-cadherin — epithelial integrity',
    'VIM': 'Vimentin — EMT marker',
    'SERPINE1': 'PAI-1 — invasion and angiogenesis',
    'MKI67': 'Ki-67 — proliferation marker',
    'CDKN2A': 'p16 — cell cycle regulator',
}

print('Known Transition Gene Status in GSE106191 (Hyperplasia vs Cancer):')
print('=' * 95)
print(f'{"Gene":<10} {"Direction":<20} {"logFC":>8} {"adj.p":>10}  {"Interpretation"}')
print('-' * 95)

check_results = []
for gene, desc in known_transition.items():
    row = deg_transition[deg_transition['Gene'] == gene]
    if len(row) > 0:
        r = row.iloc[0]
        direction = r['Direction']
        logfc = r['logFC']
        adjp = r['adj_pvalue']
        
        if direction == 'Up_in_Cancer':
            interp = 'Gained in progression'
        elif direction == 'Up_in_Hyperplasia':
            interp = 'Lost in progression'
        else:
            interp = 'Not significant'
        
        print(f'{gene:<10} {direction:<20} {logfc:>8.3f} {adjp:>10.2e}  {interp}')
    else:
        direction = 'NA'
        logfc = None
        adjp = None
        interp = 'Not in dataset'
        print(f'{gene:<10} {"Not in dataset":<20} {"":>8} {"":>10}  {desc}')
    
    check_results.append({
        'Gene': gene, 'Description': desc, 'Direction': direction,
        'logFC': logfc, 'adj_pvalue': adjp
    })

check_df = pd.DataFrame(check_results)
check_df.to_csv('results/known_transition_gene_check_gse106191.csv', index=False)

n_sig = check_df[check_df['Direction'].isin(['Up_in_Cancer', 'Up_in_Hyperplasia'])].shape[0]
n_ns = (check_df['Direction'] == 'NS').sum()
n_na = (check_df['Direction'] == 'NA').sum()
print(f'\nSummary: {n_sig} significant, {n_ns} not significant, {n_na} not in dataset')

## 10. Summary Statistics and Downstream Compatibility

In [ ]:
# Standardize direction labels for PCBS compatibility
# PCBS expects: 'Up' and 'Down' relative to disease progression
# In our convention:
#   Up_in_Cancer → gene goes UP during hyperplasia→cancer → 'Up' in progression
#   Up_in_Hyperplasia → gene goes DOWN during hyperplasia→cancer → 'Down' in progression

deg_for_pcbs = deg_transition.copy()
deg_for_pcbs['Direction_PCBS'] = 'NS'
deg_for_pcbs.loc[deg_for_pcbs['Direction'] == 'Up_in_Cancer', 'Direction_PCBS'] = 'Up'
deg_for_pcbs.loc[deg_for_pcbs['Direction'] == 'Up_in_Hyperplasia', 'Direction_PCBS'] = 'Down'

# Invert logFC so positive = upregulated in cancer progression
# (original: positive = higher in hyperplasia, but for T(g) we want progression direction)
deg_for_pcbs['logFC_progression'] = -deg_for_pcbs['logFC']

print('Direction mapping for PCBS T(g):')
print('  Up_in_Cancer → Up (upregulated during progression)')
print('  Up_in_Hyperplasia → Down (downregulated during progression)')
print(f'\nPCBS direction counts:')
print(deg_for_pcbs['Direction_PCBS'].value_counts().to_string())

In [ ]:
# Save ALL genes with limma statistics — key output for PCBS T(g)
output_cols = ['Gene', 'logFC', 'logFC_progression', 'AveExpr', 't',
               'pvalue', 'adj_pvalue', 'B', 'Direction', 'Direction_PCBS']
deg_for_pcbs[output_cols].to_csv('results/all_genes_limma_gse106191.csv', index=False)

# Also save the expression matrix
expr.to_csv('results/expr_matrix_gse106191.csv')

# Save metadata
meta_df[['GSM', 'Title', 'Group']].to_csv('results/metadata_gse106191.csv', index=False)

# Save as pickle for easy loading
import pickle
with open('results/gse106191_outputs.pkl', 'wb') as f:
    pickle.dump({
        'deg_transition': deg_transition,
        'deg_for_pcbs': deg_for_pcbs,
        'meta_df': meta_df,
        'enr_hyp': enr_hyp,
        'enr_can': enr_can,
    }, f)

# Final Summary
n_up_prog = (deg_for_pcbs['Direction_PCBS'] == 'Up').sum()
n_down_prog = (deg_for_pcbs['Direction_PCBS'] == 'Down').sum()

print('=' * 65)
print('NOTEBOOK 3 COMPLETE — GSE106191 Transition Evidence')
print('=' * 65)
print(f'\nDataset: GSE106191 (Hermens et al.)')
print(f'Platform: GPL570 (Affymetrix HG-U133 Plus 2.0)')
print(f'Samples: {len(meta_df)} ({len(cancer_samples)} Cancer, {len(hyp_samples)} Hyperplasia)')
print(f'Comparison: Hyperplasia vs Cancer (transition genes)')
print(f'Genes analyzed: {expr.shape[0]}')
print(f'\nMethod: limma (empirical-Bayes moderation)')
print(f'Filter: adj.p < 0.05, no logFC cutoff')
print(f'\nTransition DEGs: {n_up_prog + n_down_prog}')
print(f'  Upregulated in progression (Hyp→Cancer): {n_up_prog}')
print(f'  Downregulated in progression (Hyp→Cancer): {n_down_prog}')
print(f'\nKey output for PCBS T(g): results/all_genes_limma_gse106191.csv')
print(f'  Contains ALL {len(deg_for_pcbs)} genes with:')
print(f'  - logFC (Hyperplasia vs Cancer, original)')
print(f'  - logFC_progression (inverted: positive = up in cancer)')
print(f'  - adj_pvalue, Direction_PCBS (Up/Down/NS)')
print(f'\n-> Next: Notebook 4 (GSE39099 Directional Support → D(g))')

In [ ]:
# Download results
!zip -r gse106191_results_v2.zip results/ figures/
from google.colab import files
files.download('gse106191_results_v2.zip')
print('Results downloaded.')